In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Titanic Survival Machine Learning Pipeline

## Step 1: Load Dataset

In this step, we load the Titanic dataset into a pandas DataFrame and check whether the data has been imported correctly.

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [11]:
df = pd.read_csv("../data/titanic_real_world_missing_values_dataset.csv")

In [12]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [13]:
df.shape

(891, 12)

In [14]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')

In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [16]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Step 2: Initial Data Inspection

In this step, we check the dataset shape, column names, first few records, and data types. This helps us understand the structure of the dataset before cleaning or analysis.

## Step 3: Identify Target Column

In this step, we identify the target variable that the machine learning model needs to predict.

For this dataset, the target column is `Survived`.

- `0` = Passenger did not survive
- `1` = Passenger survived

This means the project is a binary classification problem.

In [17]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')

In [18]:
target_col = "Survived"

In [19]:
df[target_col].value_counts()

Survived
0    549
1    342
Name: count, dtype: int64

In [20]:
df[target_col].value_counts(normalize=True) * 100

Survived
0    61.616162
1    38.383838
Name: proportion, dtype: float64

### Step 3 Findings

The target column is `Survived`.

This is a binary classification problem because the target has two possible values:

- `0`: Passenger did not survive
- `1`: Passenger survived

The target distribution shows that more passengers did not survive than survived, so class distribution should be considered during model evaluation.

## Step 4: Check Missing Values

In this step, we check missing values in the dataset.

Missing values are important because machine learning models cannot properly learn from incomplete data. Before training a model, we need to identify which columns contain missing values and decide how to handle them.


In [21]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [22]:
missing_count = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent
})

missing_summary

,missing_count,missing_percent
PassengerId,0,0.000000
Survived,0,0.000000
Pclass,0,0.000000
Name,0,0.000000
Sex,0,0.000000
Age,177,19.865320
SibSp,0,0.000000
Parch,0,0.000000
Ticket,0,0.000000
Fare,0,0.000000


In [23]:
missing_summary[missing_summary["missing_count"] > 0]

,missing_count,missing_percent
Age,177,19.865320
Cabin,687,77.104377
Embarked,2,0.224467


In [24]:
missing_summary[missing_summary["missing_count"] > 0]

,missing_count,missing_percent
Age,177,19.865320
Cabin,687,77.104377
Embarked,2,0.224467


### Step 4 Findings

The dataset contains missing values in the following columns:

- `Age`: around 20% missing values
- `Cabin`: around 77% missing values
- `Embarked`: very few missing values

Cleaning decisions:

- `Age` will be filled using the median value because it is a useful numerical feature.
- `Cabin` will be dropped because it has too many missing values.
- `Embarked` will be filled using the mode because it is a categorical feature with very few missing values.

## Step 5: Check Duplicate Records

In this step, we check whether the dataset contains duplicate rows.

Duplicate records can affect analysis and model training because repeated rows may bias the model. Therefore, duplicates should be identified and removed if necessary.

In [25]:
df.duplicated().sum()

np.int64(0)

In [26]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


In [27]:
df = df.drop_duplicates()

In [28]:
print("Duplicate rows after cleaning:", df.duplicated().sum())
print("Dataset shape after duplicate check:", df.shape)

Duplicate rows after cleaning: 0
Dataset shape after duplicate check: (891, 12)


### Step 5 Findings

The dataset was checked for duplicate records.

Duplicate rows can create bias in analysis and model training because the same record may be counted more than once. After checking and applying duplicate removal, the dataset is ready for the next cleaning step.

## Step 6: Clean Data

In this step, we clean the dataset by handling missing values and removing columns that are not useful for the basic machine learning model.

Cleaning decisions:

- Fill missing `Age` values using the median.
- Fill missing `Embarked` values using the mode.
- Drop `Cabin` because it has too many missing values.
- Drop `PassengerId`, `Name`, and `Ticket` because they are identifiers or high-cardinality text fields that are not useful for this basic model.

In [30]:
print("Dataset shape before cleaning:", df.shape)

Dataset shape before cleaning: (891, 12)


In [31]:
age_median = df["Age"].median()
print("Age median:", age_median)

df["Age"] = df["Age"].fillna(age_median)

Age median: 28.0


In [32]:
embarked_mode = df["Embarked"].mode()[0]
print("Embarked mode:", embarked_mode)

df["Embarked"] = df["Embarked"].fillna(embarked_mode)

Embarked mode: S


In [33]:
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])


In [34]:
print("Dataset shape after cleaning:", df.shape)
print(df.isnull().sum())

Dataset shape after cleaning: (891, 8)
Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64


In [35]:
print("Dataset shape after cleaning:", df.shape)
print(df.isnull().sum())


Dataset shape after cleaning: (891, 8)
Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64


In [36]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


### Step 6 Findings

The dataset was cleaned using the following decisions:

- Missing `Age` values were filled using the median.
- Missing `Embarked` values were filled using the mode.
- `Cabin` was removed because it had a high percentage of missing values.
- `PassengerId`, `Name`, and `Ticket` were removed because they are not useful for the basic machine learning model.

After cleaning, the dataset has no missing values and is ready for exploratory data analysis.